# Limpieza y Analisis a detalle de la base de datos


In [1]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")  # subir un nivel

from scripts import bases as b, init_notebook as init, limpieza as lim

ctx = init.init_notebook()

df_pacientes = ctx["df_pacientes"]

Cargando datos principales...
Pacientes y traslados cargados.
Cargando datos geográficos...
Datos geográficos cargados.


In [3]:
df_pacientes.head(5)

,Id Hospital,Nombre Hospital,Id,Fecha inicio,Estado al ingreso,Tipo al ingreso,Último estado,Último tipo,Sexo,Edad,...,Asistencia Respiratoria Mecánica,Motivo,Operación,Fecha egreso,Última actualización,Pasó por Críticas,Pasó por Intermedias,Pasó por Generales,Duracion días,murio
0,1,EL CRUCE,Mariano,NaT,sospechosos,criticas,sospechosos,criticas,NaN,NaN,...,NaN,NaN,NaN,NaT,2020-07-21 14:34:40,si,no,no,NaN,False
1,1,EL CRUCE,12345,NaT,sospechosos,intermedias,sospechosos,criticas,NaN,40,...,NaN,NaN,NaN,NaT,2020-07-21 14:34:40,si,si,no,NaN,False
2,1,EL CRUCE,24698750,2020-05-10 12:11:05,sospechosos,criticas,sospechosos,criticas,femenino,44,...,NaN,NaN,NaN,NaT,2020-07-21 14:34:40,si,no,no,NaN,False
3,1,EL CRUCE,1001,2020-05-28 12:11:05,ocupadas_covid,intermedias,ocupadas_covid,intermedias,masculino,49,...,NaN,alta-domiciliaria,egreso,2020-06-09 12:11:05,2020-07-21 14:34:40,no,si,no,12.0,False
4,1,EL CRUCE,1002,2020-05-15 12:11:05,ocupadas_covid,criticas,ocupadas_covid,intermedias,femenino,NaN,...,NaN,NaN,NaN,NaT,2020-07-21 14:34:40,si,si,no,NaN,False


In [ ]:

df_pacientes = lim.limpiar_dataset_pro(df_pacientes)

traslados = ctx["traslados"]
hosp_coords = ctx["hosp_coords"]
municipios = ctx["municipios"]
municipios_amba = ctx["municipios_amba"]

In [ ]:
df_pacientes.groupby("paciente_id").nunique()

In [ ]:
df_pacientes.columns

In [ ]:
df_pacientes["paciente_id"].nunique(), len(df_pacientes)

In [ ]:
# =============================================================================
# IMPORTANTE: reconstruir trayectoria por paciente
# (porque hay multiples filas por id debido a traslados)
# =============================================================================

# mapear niveles de cuidado
mapa = {
    "generales": 0,
    "intermedias": 1,
    "criticas": 2
}

df_pacientes["nivel"] = df_pacientes["tipo_ingreso"].map(mapa)

# ordenar por paciente y tiempo (ajustar nombre de fecha si es necesario)
df_pacientes = df_pacientes.sort_values(["paciente_id", "fecha_inicio"])

# resumen por paciente
df_resumen = df_pacientes.groupby("paciente_id").agg(
    tipo_ingreso=("tipo_ingreso", "first"),
    nivel_inicial=("nivel", "first"),
    nivel_max=("nivel", "max"),
    paso_criticas=("nivel", lambda x: (x == 2).any()),
    fallecio=("fallecio", "max")
).reset_index()

# evolucion real del paciente
df_resumen["evolucion"] = df_resumen["nivel_max"] - df_resumen["nivel_inicial"]

In [ ]:
# =============================================================================
# ANALISIS
# =============================================================================

# ¿Cuál es la probabilidad de morir según el nivel de ingreso?
print(df_resumen.groupby("tipo_ingreso")["fallecio"].mean())

print(df_resumen.groupby("tipo_ingreso")["fallecio"].agg(["mean", "count", "sum"]))

"""
OBSERVACIONES:

1. grandes diferencias en mortalidad según nivel inicial
2. pacientes críticos tienen mayor riesgo individual
3. la mayoría de muertes proviene de intermedias (mayor volumen)

PRIMERA CONCLUSION:
Aunque los pacientes críticos tienen mayor riesgo individual,
la mayoría de las muertes ocurre en pacientes que ingresan a unidades
intermedias, debido a su mayor volumen.
"""

In [ ]:
df_resumen.iloc[((df_resumen["evolucion"] > 0) )]

In [ ]:
# =============================================================================
# HIPOTESIS
# =============================================================================

# H1: empeoran y pasan a críticas
print(
    df_resumen[
        (df_resumen["tipo_ingreso"] == "intermedias") &
        (df_resumen["evolucion"] > 0) &
        (df_resumen["paso_criticas"])
    ]["fallecio"].mean()
)

# H2: empeoran pero NO pasan a críticas
print(
    df_resumen[
        (df_resumen["tipo_ingreso"] == "intermedias") &
        (df_resumen["evolucion"] > 0) &
        (~df_resumen["paso_criticas"])
    ]["fallecio"].mean()
)

# H3: posible mala clasificación inicial
print(
    df_resumen[
        (df_resumen["tipo_ingreso"] == "intermedias") &
        (df_resumen["evolucion"] == 0) &
        (~df_resumen["paso_criticas"])
    ]["fallecio"].mean()
)

In [ ]:
df_resumen.iloc[(df_resumen["evolucion"]==1)]

In [ ]:
# =============================================================================
# ANALISIS DE EVOLUCION
# =============================================================================

print(df_resumen.groupby("evolucion")["fallecio"].mean())

print(
    df_resumen.groupby(["tipo_ingreso", "evolucion"])["fallecio"]
    .agg(["mean", "count"])
)

"""
SEGUNDA CONCLUSION:
La evolución negativa es un factor muchísimo más discriminante
que el estado inicial.
"""

In [ ]:
# =============================================================================
# TABLA RESUMEN FINAL
# =============================================================================

tabla_final = df_resumen.groupby(
    ["tipo_ingreso", "evolucion"]
)["fallecio"].agg(
    mortalidad="mean",
    pacientes="count"
).reset_index()

print(tabla_final)

In [ ]:
# =============================================================================
# GRAFICO FINAL (claro y robusto)
# =============================================================================

# categorizar evolucion para simplificar
def categorizar(e):
    if e < 0:
        return "mejora"
    elif e == 0:
        return "igual"
    else:
        return "empeora"

df_resumen["evol_cat"] = df_resumen["evolucion"].apply(categorizar)

tabla_plot = df_resumen.groupby(
    ["tipo_ingreso", "evol_cat"]
)["fallecio"].mean().reset_index()

plt.figure()

sns.barplot(
    data=tabla_plot,
    x="tipo_ingreso",
    y="fallecio",
    hue="evol_cat"
)

plt.ylabel("Probabilidad de fallecer")
plt.xlabel("Tipo de ingreso")
plt.title("Impacto de la evolución en la mortalidad")
plt.show()

In [ ]:
# # exploracion
# lim.explorar_valores(df_pacientes)

# # debug / calidad de datos
# lim.detectar_problemas(df_pacientes)
# lim.check_coherencia(df_pacientes)

# # check
# lim.check_post_limpieza(df_pacientes)

In [ ]:
# Riesgo por nivel inicial 
# ¿Cuál es la probabilidad de morir según el nivel de gravedad al ingreso?
df_pacientes.groupby("tipo_ingreso")["fallecio"].mean()

# Empeorar culmino con fallecimiento
# ¿Empeorar dentro del hospital está asociado con mayor mortalidad?
# ojo: sesgo de supervivencia --> si alguien muere rápido → no “evoluciona”
df_pacientes.groupby("evolucion")["fallecio"].mean()

# Pacientes criticos
# ¿Los pacientes que alguna vez estuvieron en UTI tienen mayor mortalidad?
# ojo: mezcla pacientes graves desde el inicio con pacientes que empeoraron
df_pacientes[df_pacientes["paso_criticas"]]["fallecio"].mean()


## aca trabajamos con 3 cosas distintas, 3 variables:
# | variable      | qué mide                   |
# | ------------- | -------------------------- |
# | tipo_ingreso  | gravedad inicial           |
# | evolucion     | cambio dentro del hospital |
# | paso_criticas | evento crítico             |


In [ ]:
print(df_pacientes.groupby("tipo_ingreso")["fallecio"].mean())
# agrupa pacientes por EL NIVEL DE GRAVEDAD CON EL CUAL entraron
# calcula el promedio de fallecio (True=1, False=0). PUEDE SER TAMBIEN PROBABILIDAD

"""
falta contexto:
¿cuántos pacientes hay en cada grupo?
¿qué proporción representan?
"""

print(df_pacientes.groupby("tipo_ingreso")["fallecio"].agg(["mean", "count", "sum"]))

# 1. obs:
# muchisimas diferencias. 35% vs 7% vs 2.5%.
# LOS QUE ENTRAN EN CRITICO ya estan fuertemente asociados con el riesgo de morir


# 2. obs: generales tiene poca muestra, porque hay pocas muertes pero pocos casos
# criticos tiene pocos casos pero muchas muertes asi que es medio confiable
# intermedias muy confiable


# 3. obs: muchas muertes vienen de intermedios


"""
PRIMERA CONCLUSION:
Aunque los pacientes críticos tienen mayor riesgo individual,
la mayoría de las muertes ocurre en pacientes que ingresan a unidades
intermedias, debido a su mayor volumen.
"""

# ¿Por qué mueren tantos pacientes intermedios?
# H1: empeoran y pasan a críticas
# H2: no llegan a ser transferidos
# H3: mala clasificación inicial


## hipotesis 1 ##
print(df_pacientes[df_pacientes["tipo_ingreso"] == "intermedias"] \
     .groupby("evolucion")["fallecio"].mean())

# 3% vs 4.5% vs SETENTA Y TRES %. ES MUCHISIMA DIFERENCIA !
df_pacientes.groupby(["tipo_ingreso", "evolucion"])["fallecio"].mean()


"""
El impacto de una evolución negativa no es homogéneo:
pacientes inicialmente no críticos experimentan aumentos drásticos
en la mortalidad al empeorar, alcanzando niveles comparables
a pacientes críticos.
"""

df_pacientes.groupby(["tipo_ingreso", "evolucion"])["fallecio"].count()


"""
SEGUNDA CONCLUSION:
La evolución negativa es un factor muchísimo más discriminante
que el estado inicial.
"""

In [ ]:
# resumen de lo hecho en el bloque anterior:

tabla_final = df_pacientes.groupby(
    ["tipo_ingreso", "evolucion"]
)["fallecio"].agg(
    mortalidad="mean",
    pacientes="count"
).reset_index()

print(tabla_final)


# grafico



tabla = df_pacientes.groupby(
    ["tipo_ingreso", "evolucion"]
)["fallecio"].mean().reset_index()

# filtrar extremos con poca muestra (opcional pero recomendado)
counts = df_pacientes.groupby(
    ["tipo_ingreso", "evolucion"]
)["fallecio"].count().reset_index(name="count")

tabla = tabla.merge(counts, on=["tipo_ingreso", "evolucion"])
tabla = tabla[tabla["count"] > 50] # sacamos si hay menos de 50 casos

plt.figure()

sns.barplot(
    data=tabla,
    x="tipo_ingreso",
    y="fallecio",
    hue="evolucion"
)

plt.ylabel("Probabilidad de fallecer")
plt.xlabel("Tipo de ingreso")
plt.title("Mortalidad según ingreso y evolución")

plt.legend(title="Evolución")
plt.show()

## 1. Funcionamiento de la Red General (AMBA)

## 2. Trayectorias de Paciente

### 2.1 Cantidad de traslados para cada persona (promedio y desvío, junto a distribución)

In [ ]:
# Distribución de traslados por paciente
conteo_tras_paciente, stats_tras_paciente = bases.distribucion_traslados_paciente(traslados, col_id="Id", valores=[1, 2, 3, 4, 5, 6, 7], graficar=True)

In [ ]:
## que paso con estos 3?
## reconstruir estos 3

# incluir los ingresos como porcentaje. escala logaritmica

# PROBABILIDAD DE TENER UN TRASLADO MAS DADO LA CANTIDAD DE TRASLADOS QUE TUVISTE

# Cuantos de esos tambien tienen como lo que tuviste

In [ ]:
#bases.revisar_dias_negativos(traslados, max_pacientes=10)

In [ ]:
# # pacientes con error
# errores = traslados[traslados["error_fecha"]]
# print("Pacientes con errores de fechas:", errores["Id"].unique())

# # historial completo de un paciente con error
# # usar el DataFrame de traslados que tiene las columnas calculadas
# pid = errores["Id"].iloc[1]

# historial = traslados[traslados["Id"] == pid].sort_values("Fecha inicio")

# display(historial[[
#     "Nombre Hospital",
#     "Fecha inicio",
#     "Fecha egreso",
#     "Hospital siguiente",
#     "Fecha ingreso siguiente",
#     "dias_entre_hospitales",
#     "error_fecha",
#     "gravedad_error"
# ]])

In [ ]:
# pacientes con >=3 traslados
traslados_muchos = bases.pacientes_con_muchos_traslados(traslados, minimo=3)

# imprimir recorridos
# bases.imprimir_recorridos_pacientes(
#     traslados_muchos,
#     col_id="Id",
#     col_origen="Nombre Hospital",
#     col_destino="Hospital siguiente"
# )

In [ ]:
def detectar_rebotes(df, umbral_dias=1):
    
    rebotes = df[
        (df["dias_entre_hospitales"] >= 0) &
        (df["dias_entre_hospitales"] <= umbral_dias)
    ]
    
    print("Cantidad de rebotes:", len(rebotes))
    
    display(
        rebotes[[
            "Id",
            "Nombre Hospital",
            "Hospital siguiente",
            "dias_entre_hospitales"
        ]].head(10)
    )
    
    return rebotes

rebotes=detectar_rebotes(traslados)
df_pacientes.head(5)

In [ ]:
#bases.graficar_estado_paciente_debug(traslados_muchos.head(5))

In [ ]:
# # ------------------------------
# # EJECUCIÓN EN EL NOTEBOOK
# # ------------------------------
# # Mostrar tablas por paciente
# bases.mostrar_recorridos_estado(traslados_muchos.head(9))

# # Sankey de flujo
# bases.sankey_pacientes(traslados_muchos)

### 2.2 Tiempo dentro del sistema por persona

In [ ]:
# Tiempo en el sistema por persona
#tiempo_sis, limite_tiempo = bases.tiempo_total_paciente(df_pacientes, col_id="Id", col_dias="Duracion días", max_dias=100, quantile_outlier=0.95, graficar=True)

In [ ]:
# print("n pacientes:", len(tiempo_sis))
# print("min:", tiempo_sis.min())
# print("max:", tiempo_sis.max())
# print("valores:", tiempo_sis.value_counts().head(10))

In [ ]:
# un grafico grande y te vas quedando con info
# ir bajandolo a temas

## 3. Análisis Descriptivo por Hospital

### 3.1 Traslados por hospital

In [ ]:
# # Traslados OUT (Origen)
# traslados_out = bases.traslados_por_hospital(traslados, col_hospital="Nombre Hospital", graficar=False)

# # Traslados IN (Destino)
# traslados_in = bases.traslados_por_hospital(traslados, col_hospital="Hospital siguiente", graficar=False)

# tabla_hospitales = pd.DataFrame({
#     "traslados_out": traslados_out,
#     "traslados_in": traslados_in,
# }).fillna(0)

# print("Top 10 hospitales que derivan más pacientes:")
# display(tabla_hospitales.sort_values(by="traslados_out", ascending=False).head(10))

# print("\nTop 10 hospitales que reciben más pacientes:")
# display(tabla_hospitales.sort_values(by="traslados_in", ascending=False).head(10))

# # Graficamos los IN
# tabla_hospitales["traslados_in"].sort_values(ascending=False).head(10).plot(kind="bar", color="skyblue", figsize=(10,5))
# plt.title("Top 10 Hospitales por Ingresos (Traslados IN)")
# plt.ylabel("Cantidad de traslados recibidos")
# plt.xlabel("Hospital")
# plt.xticks(rotation=45, ha='right')
# plt.show()

In [ ]:
# ESTO ES STRENGTH O FUERZA
# DEGREE: # de hospitales con los que me conecto

# fuerza/grado = numero promedio de traslados que recibo de un hospital con el que estoy conectado

### 3.2 Tiempo promedio que pasa una persona dentro del hospital

In [ ]:
# # Tiempo promedio por hospital
# tiempo_prom_hosp = bases.tiempo_promedio_por_hospital(df_pacientes, col_hospital="Nombre Hospital", col_dias="Duracion días", quantile_outlier=0.95, graficar=False)

# display(tiempo_prom_hosp.head(10))

# tiempo_prom_hosp.head(15).plot(kind="bar", color="orange", figsize=(10,5))
# plt.title("Top 15 Hospitales por Tiempo Promedio de Estadía")
# plt.ylabel("Días promedio")
# plt.xlabel("Hospital")
# plt.xticks(rotation=45, ha='right')
# plt.show()

In [ ]:
# agregar los valores a las barras

### 3.3 Cantidad de muertos por hospital

In [ ]:
# muertes_hosp = bases.muertes_por_hospital(
#     df_pacientes,
#     col_hospital="Nombre Hospital",
#     col_muerte="murio",
#     graficar=False
# )

# display(muertes_hosp.head(10))

# ax = muertes_hosp.head(15).plot(kind="bar", color="red", figsize=(10,5))

# # agregar numerito arriba de cada barra
# for p in ax.patches:
#     height = p.get_height()
#     ax.text(
#         p.get_x() + p.get_width()/2,
#         height,
#         str(int(height)),
#         ha="center",
#         va="bottom",
#         fontsize=9
#     )

# plt.title("Top 15 Hospitales por Cantidad de Fallecidos")
# plt.ylabel("Cantidad de fallecidos")
# plt.xlabel("Hospital")
# plt.xticks(rotation=45, ha='right')
# plt.show()

In [ ]:
## entrar a la red o volver a casa o muerte
## version: optimista


# ver que rol tienen estos indicadores en la caminata

In [ ]:
## ver los cambios de estado de las personas
## cuanta gente pasa de estado general a critico, intermedio a general, etc etc

## traslados entre hospitales  y traslados entre niveles
# la gente mejora o empeora en el hospital? cuantos?

## 4. Análisis Combinado

### 4.1 Cantidad de personas con distintos niveles de riesgo social y estados (crítico, intermedio general)

In [ ]:
# # Tabla cruzada de Nivel de riesgo social y Estado al ingreso
# # Tabla cruzada de Nivel de riesgo social y Estado al ingreso
# if "Nivel riesgo social" in df_pacientes.columns and "Estado al ingreso" in df_pacientes.columns:

#     tabla_riesgo_estado = pd.crosstab(
#         df_pacientes["Nivel riesgo social"],
#         df_pacientes["Estado al ingreso"],
#         margins=True,
#         margins_name="Total"
#     )

#     display(tabla_riesgo_estado)

#     tabla_sin_totales = pd.crosstab(
#         df_pacientes["Nivel riesgo social"],
#         df_pacientes["Estado al ingreso"]
#     )

#     tabla_sin_totales.plot(
#         kind="bar",
#         stacked=False,
#         figsize=(10,6),
#         colormap="viridis"
#     )

#     plt.title("Estado al ingreso según Nivel de Riesgo Social")
#     plt.ylabel("Cantidad de pacientes")
#     plt.xlabel("Nivel de Riesgo")
#     plt.xticks(rotation=0)
#     plt.legend(title="Estado al ingreso")
#     plt.tight_layout()
#     plt.show()

# else:
#     print("Las columnas necesarias para este análisis no están disponibles.")
#     print("Columnas disponibles:", df_pacientes.columns.tolist())